In [ ]:
"""
==============================================================================
 LSTM MULTIVARIADA por centro — Predicción de carga parasitaria
==============================================================================
 Red LSTM que predice el total de parásitos del día siguiente a partir de
 una ventana de los 14 días previos de las variables físicas y la
 estacionalidad. Se entrena un modelo por centro, por separado.

 Dos variantes de estacionalidad, para comparar:
   - "estacion" : estación en one-hot (4 columnas)
   - "ciclica"  : codificación cíclica del día del año (seno/coseno)

 Requiere: pip install tensorflow scikit-learn

 BLOQUES:
   1 Configuración   2 Carga + features   3 Secuencias (sin cruzar huecos)
   4 Modelo LSTM     5 Entrenamiento/evaluación   6 Reporte   7 Main
==============================================================================
"""

import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

# TensorFlow / Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Total_Parasitos_",
    "fisicas"    : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "col_centro" : "id_centro",
    "col_fecha"  : "Fecha",
    "col_estacion": "Estacion",

    "ventana"    : 10,           # días hacia atrás por secuencia (reducida)
    "test_frac"  : 0.20,         # último 20% temporal como test
    "estacionalidad": "estacion",  # "estacion" (one-hot) o "ciclica"

    # Arquitectura e hiperparámetros de la red (simplificada para pocos datos)
    "unidades"   : 16,           
    "dropout"    : 0.5,          
    "epocas"     : 150,
    "batch"      : 16,
    "paciencia"  : 15,           # early stopping
}


# ============================================================================
# BLOQUE 2 — CARGA Y CONSTRUCCIÓN DE FEATURES
# ============================================================================
def cargar_centro(config, centro):
    df = pd.read_csv(config["ruta_csv"])
    df[config["col_fecha"]] = pd.to_datetime(df[config["col_fecha"]])
    df = df[df[config["target"]].notna()]
    sub = df[df[config["col_centro"]] == centro].sort_values(
        config["col_fecha"]).reset_index(drop=True)

    features = list(config["fisicas"])

    # Estacionalidad: one-hot o cíclica
    if config["estacionalidad"] == "estacion":
        dummies = pd.get_dummies(sub[config["col_estacion"]], prefix="Est")
        sub = pd.concat([sub, dummies], axis=1)
        features += list(dummies.columns)
    elif config["estacionalidad"] == "ciclica":
        dia_anio = sub[config["col_fecha"]].dt.dayofyear
        sub["dia_sin"] = np.sin(2 * np.pi * dia_anio / 365.25)
        sub["dia_cos"] = np.cos(2 * np.pi * dia_anio / 365.25)
        features += ["dia_sin", "dia_cos"]

    return sub, features


# ============================================================================
# BLOQUE 3 — CONSTRUCCIÓN DE SECUENCIAS
# ============================================================================
def construir_secuencias(config, sub, features):
    fechas = sub[config["col_fecha"]].values
    Xmat = sub[features].astype(float).values
    yvec = sub[config["target"]].astype(float).values
    v = config["ventana"]

    X_seq, y_seq, idx_fin = [], [], []
    for i in range(len(sub) - v):
        # Verificar que los v+1 días sean consecutivos (sin hueco)
        bloque = fechas[i:i + v + 1]
        saltos = np.diff(bloque).astype("timedelta64[D]").astype(int)
        if np.all(saltos == 1):                 # solo secuencias continuas
            X_seq.append(Xmat[i:i + v])
            y_seq.append(yvec[i + v])           # predice el día siguiente
            idx_fin.append(i + v)
    return np.array(X_seq), np.array(y_seq), np.array(idx_fin)


# ============================================================================
# BLOQUE 4 — MODELO LSTM
# ============================================================================
def construir_modelo(config, n_features):
    modelo = Sequential([
        LSTM(config["unidades"], input_shape=(config["ventana"], n_features)),
        Dropout(config["dropout"]),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    modelo.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return modelo


# ============================================================================
# BLOQUE 5 — ENTRENAMIENTO Y EVALUACIÓN (validación temporal)
# ============================================================================
def entrenar_evaluar(config, X_seq, y_seq):
    # División temporal: primeras secuencias train, últimas test
    n = len(X_seq)
    corte = int(n * (1 - config["test_frac"]))
    Xtr, Xte = X_seq[:corte], X_seq[corte:]
    ytr, yte = y_seq[:corte], y_seq[corte:]

    # Normalización de features, ajustada SOLO con train (evita fuga)
    n_feat = Xtr.shape[2]
    scaler_X = StandardScaler()
    Xtr_2d = Xtr.reshape(-1, n_feat)
    scaler_X.fit(Xtr_2d)
    Xtr = scaler_X.transform(Xtr_2d).reshape(Xtr.shape)
    Xte = scaler_X.transform(Xte.reshape(-1, n_feat)).reshape(Xte.shape)

    # Normalización del TARGET, también solo con train. Las LSTM entrenan
    # mucho mejor cuando la variable objetivo está en escala pequeña; las
    # predicciones se devuelven luego a la escala original para las métricas.
    scaler_y = StandardScaler()
    ytr_s = scaler_y.fit_transform(ytr.reshape(-1, 1)).flatten()

    modelo = construir_modelo(config, n_feat)
    es = EarlyStopping(monitor="val_loss", patience=config["paciencia"],
                       restore_best_weights=True)
    modelo.fit(Xtr, ytr_s, validation_split=0.15,
               epochs=config["epocas"], batch_size=config["batch"],
               callbacks=[es], verbose=0)

    # Predecir en escala normalizada y devolver a la escala original
    pred_s = modelo.predict(Xte, verbose=0).flatten()
    pred = scaler_y.inverse_transform(pred_s.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(yte, pred))
    return {
        "n_train": len(Xtr), "n_test": len(Xte),
        "MAE" : mean_absolute_error(yte, pred),
        "RMSE": rmse,
        "R2"  : r2_score(yte, pred),
    }


# ============================================================================
# BLOQUE 6 — REPORTE
# ============================================================================
def reportar(centro, res, config):
    print("=" * 60)
    print(f"  CENTRO {centro} — LSTM ({config['estacionalidad']})")
    print("=" * 60)
    print(f"  Ventana: {config['ventana']} días | "
          f"train={res['n_train']} test={res['n_test']}")
    print(f"  MAE : {res['MAE']:.4f}")
    print(f"  RMSE: {res['RMSE']:.4f}")
    print(f"  R²  : {res['R2']:.4f}")
    print()


# ============================================================================
# BLOQUE 7 — MAIN
# ============================================================================
def main():
    df = pd.read_csv(CONFIG["ruta_csv"])
    centros = df[df[CONFIG["target"]].notna()][CONFIG["col_centro"]].unique()

    print(f"\nLSTM multivariada — estacionalidad: {CONFIG['estacionalidad']}\n")
    for centro in centros:
        sub, features = cargar_centro(CONFIG, centro)
        X_seq, y_seq, _ = construir_secuencias(CONFIG, sub, features)
        if len(X_seq) < 50:
            print(f"  Centro {centro}: muy pocas secuencias ({len(X_seq)}), omitido")
            continue
        res = entrenar_evaluar(CONFIG, X_seq, y_seq)
        reportar(centro, res, CONFIG)


if __name__ == "__main__":
    main()

In [ ]:
"""
==============================================================================
 LSTM VALOR ABSOLUTO + TODOS LOS CENTROS JUNTOS
==============================================================================

 Requiere: pip install tensorflow scikit-learn


 BLOQUES:
   1 Config  2 Carga+features  3 Secuencias  4 Modelo
   5 Entrenamiento/evaluación  6 Main
==============================================================================
"""

import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"   : "data/Dataset_Maestro_Amplificado_ML.csv",
    "target"     : "Total_Parasitos_",
    "fisicas"    : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "col_centro" : "id_centro",
    "col_fecha"  : "Fecha",
    "col_estacion": "Estacion",

    "ventana"    : 10,
    "test_frac"  : 0.20,
    "estacionalidad": "ciclica",   # "estacion" o "ciclica"

    "unidades"   : 24,
    "dropout"    : 0.3,
    "epocas"     : 150,
    "batch"      : 32,
    "paciencia"  : 15,
}


# ============================================================================
# BLOQUE 2 — CARGA Y FEATURES (todos los centros, con identidad de centro)
# ============================================================================
def cargar_todos(config):
    df = pd.read_csv(config["ruta_csv"])
    df[config["col_fecha"]] = pd.to_datetime(df[config["col_fecha"]])
    df = df[df[config["target"]].notna()].copy()

    features = list(config["fisicas"])

    if config["estacionalidad"] == "estacion":
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)
    elif config["estacionalidad"] == "ciclica":
        dia = df[config["col_fecha"]].dt.dayofyear
        df["dia_sin"] = np.sin(2 * np.pi * dia / 365.25)
        df["dia_cos"] = np.cos(2 * np.pi * dia / 365.25)
        features += ["dia_sin", "dia_cos"]


    centro_dum = pd.get_dummies(df[config["col_centro"]], prefix="Centro")
    df = pd.concat([df, centro_dum], axis=1)
    features += list(centro_dum.columns)

    return df, features


# ============================================================================
# BLOQUE 3 — SECUENCIAS
# ============================================================================
def construir_secuencias(config, df, features):
    v = config["ventana"]
    X_seq, y_abs, fechas_fin = [], [], []

    for centro in df[config["col_centro"]].unique():
        sub = df[df[config["col_centro"]] == centro].sort_values(
            config["col_fecha"]).reset_index(drop=True)
        fechas = sub[config["col_fecha"]].values
        Xmat = sub[features].astype(float).values
        yvec = sub[config["target"]].astype(float).values

        for i in range(len(sub) - v):
            bloque = fechas[i:i + v + 1]
            saltos = np.diff(bloque).astype("timedelta64[D]").astype(int)
            if np.all(saltos == 1):
                X_seq.append(Xmat[i:i + v])
                y_abs.append(yvec[i + v])           
                fechas_fin.append(fechas[i + v])
    return np.array(X_seq), np.array(y_abs), np.array(fechas_fin)


# ============================================================================
# BLOQUE 4 — MODELO
# ============================================================================
def construir_modelo(config, n_features):
    modelo = Sequential([
        LSTM(config["unidades"], input_shape=(config["ventana"], n_features)),
        Dropout(config["dropout"]),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    modelo.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return modelo


# ============================================================================
# BLOQUE 5 — ENTRENAMIENTO Y EVALUACIÓN (validación temporal)
# ============================================================================
def entrenar_evaluar(config, X_seq, y_abs, fechas_fin):

    orden = np.argsort(fechas_fin)
    X_seq, y_abs = X_seq[orden], y_abs[orden]

    n = len(X_seq)
    corte = int(n * (1 - config["test_frac"]))
    Xtr, Xte = X_seq[:corte], X_seq[corte:]
    ytr, yte = y_abs[:corte], y_abs[corte:]

    # Normalizar features (solo train)
    n_feat = Xtr.shape[2]
    sx = StandardScaler(); sx.fit(Xtr.reshape(-1, n_feat))
    Xtr = sx.transform(Xtr.reshape(-1, n_feat)).reshape(Xtr.shape)
    Xte = sx.transform(Xte.reshape(-1, n_feat)).reshape(Xte.shape)

    # Normalizar target (solo train)
    sy = StandardScaler()
    ytr_s = sy.fit_transform(ytr.reshape(-1, 1)).flatten()

    modelo = construir_modelo(config, n_feat)
    es = EarlyStopping(monitor="val_loss", patience=config["paciencia"],
                       restore_best_weights=True)
    modelo.fit(Xtr, ytr_s, validation_split=0.15,
               epochs=config["epocas"], batch_size=config["batch"],
               callbacks=[es], verbose=0)

    pred_s = modelo.predict(Xte, verbose=0).flatten()
    pred = sy.inverse_transform(pred_s.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(yte, pred))
    return {
        "n_train": len(Xtr), "n_test": len(Xte),
        "MAE" : mean_absolute_error(yte, pred),
        "RMSE": rmse,
        "R2"  : r2_score(yte, pred),
    }


# ============================================================================
# BLOQUE 6 — MAIN
# ============================================================================
def main():
    print("=" * 62)
    print("  LSTM VALOR ABSOLUTO + TODOS LOS CENTROS JUNTOS")
    print(f"  Estacionalidad: {CONFIG['estacionalidad']} | "
          f"ventana: {CONFIG['ventana']}")
    print("=" * 62)
    df, features = cargar_todos(CONFIG)
    X_seq, y_abs, fechas_fin = construir_secuencias(CONFIG, df, features)
    print(f"  Total secuencias (3 centros): {len(X_seq)}")
    print(f"  Features por paso: {len(features)}")
    res = entrenar_evaluar(CONFIG, X_seq, y_abs, fechas_fin)
    print(f"\n  Resultado (carga absoluta):")
    print(f"    train={res['n_train']} test={res['n_test']}")
    print(f"    MAE : {res['MAE']:.4f}")
    print(f"    RMSE: {res['RMSE']:.4f}")
    print(f"    R²  : {res['R2']:.4f}")
    print("=" * 62)


if __name__ == "__main__":
    main()